# CS-308-MJ-P Data Science and Analytics
## Slip 12

**TYBSc (Computer Science) | Sem V**

Install required libraries if missing (run once):

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy mlxtend matplotlib-venn wordcloud squarify

---
## Q1. Titanic Passenger-Class Histogram and Survival Pie Chart

> **Dataset:** `titanic.csv` — this cell creates the CSV automatically (sample data) if it is not present. If you have the real file in the same folder, the code uses that instead.

In [ ]:
# ---- Sample dataset: CSV is created automatically if it is not in the folder ----
import os
import numpy as np
import pandas as pd
if not os.path.exists('titanic.csv'):
    rng = np.random.default_rng(42)
    n = 500
    pclass = rng.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55])
    sex = rng.choice(["male", "female"], n, p=[0.64, 0.36])
    p = 0.15 + (sex == "female") * 0.5 + (pclass == 1) * 0.2 - (pclass == 3) * 0.05
    pd.DataFrame({"PassengerId": range(1, n + 1),
        "Survived": (rng.random(n) < p).astype(int), "Pclass": pclass,
        "Name": [f"Passenger {i}" for i in range(1, n + 1)], "Sex": sex,
        "Age": rng.integers(1, 80, n), "SibSp": rng.integers(0, 4, n),
        "Parch": rng.integers(0, 3, n),
        "Fare": np.round(rng.gamma(2, 15, n) * (4 - pclass), 2),
        "Embarked": rng.choice(["S", "C", "Q"], n, p=[0.72, 0.19, 0.09])}).to_csv('titanic.csv', index=False)
    print('titanic.csv', 'created')
# ---- Your program starts here ----

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("titanic.csv")
print(df.head())
class_col = "Pclass"
survival_col = "Survived"
if class_col not in df.columns or survival_col not in df.columns:
    raise ValueError("The file should contain Pclass and Survived columns.")
df[class_col].value_counts().sort_index().plot(kind="bar", color="steelblue")
plt.xlabel("Passenger Class")
plt.ylabel("Number of Passengers")
plt.title("Passengers in Each Class")
plt.tight_layout()
plt.show()
df[survival_col].value_counts().sort_index().plot(kind="pie", autopct="%1.1f%%",
                                                   labels=["Did not survive", "Survived"])
plt.ylabel("")
plt.title("Titanic Survival Distribution")
plt.tight_layout()
plt.show()

---
## Q2. Polynomial Regression for Song Popularity

> **Dataset:** `Spotify -Raw Data.csv` — this cell creates the CSV automatically (sample data) if it is not present. If you have the real file in the same folder, the code uses that instead.

In [ ]:
# ---- Sample dataset: CSV is created automatically if it is not in the folder ----
import os
import numpy as np
import pandas as pd
if not os.path.exists('Spotify -Raw Data.csv'):
    rng = np.random.default_rng(42)
    n = 500
    d = np.round(rng.uniform(0.2, 0.95, n), 3)
    pd.DataFrame({"track_name": [f"Song {i}" for i in range(n)],
        "artist": rng.choice(["Artist A", "Artist B", "Artist C", "Artist D"], n),
        "danceability": d, "energy": np.round(rng.uniform(0.1, 1, n), 3),
        "tempo": np.round(rng.uniform(60, 200, n), 1),
        "popularity": (20 + 120 * d - 100 * d ** 2 + rng.normal(0, 6, n)).clip(0, 100).round(0).astype(int)
        }).to_csv('Spotify -Raw Data.csv', index=False)
    print('Spotify -Raw Data.csv', 'created')
# ---- Your program starts here ----

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

df = pd.read_csv("Spotify -Raw Data.csv")
print(df.head())
features = ["danceability", "energy", "tempo"]
target = "popularity"
if not all(c in df.columns for c in features + [target]):
    raise ValueError("Adjust features and target to match the Spotify dataset's column names.")
df = df[features + [target]].dropna()
X = df[["danceability"]]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("R2 Score:", r2_score(y_test, pred))
order = np.argsort(X_test["danceability"].values)
plt.scatter(X_test["danceability"], y_test, alpha=0.5, label="Actual")
plt.plot(X_test["danceability"].values[order], pred[order], color="red", label="Polynomial curve")
plt.xlabel("Danceability")
plt.ylabel("Popularity")
plt.title("Polynomial Regression for Song Popularity")
plt.legend()
plt.tight_layout()
plt.show()

---
## Q2 OR. Spambase KNN Classification

> **Dataset:** `spambase.csv` — this cell creates the CSV automatically (sample data) if it is not present. If you have the real file in the same folder, the code uses that instead.

In [ ]:
# ---- Sample dataset: CSV is created automatically if it is not in the folder ----
import os
import numpy as np
import pandas as pd
if not os.path.exists('spambase.csv'):
    rng = np.random.default_rng(42)
    n = 600
    spam = (rng.random(n) < 0.4).astype(int)
    def col(ham, sp):
        return np.round(np.where(spam == 1, rng.exponential(sp, n), rng.exponential(ham, n)), 3)
    pd.DataFrame({"word_freq_make": col(0.1, 0.2), "word_freq_address": col(0.2, 0.2),
        "word_freq_all": col(0.2, 0.5), "word_freq_free": col(0.07, 0.5),
        "word_freq_money": col(0.05, 0.4), "word_freq_you": col(1.2, 2.3),
        "char_freq_exclamation": col(0.1, 0.5), "char_freq_dollar": col(0.01, 0.2),
        "capital_run_length_average": col(2.4, 9.5),
        "capital_run_length_longest": col(18, 100).round(0),
        "capital_run_length_total": col(160, 470).round(0),
        "class": spam}).to_csv('spambase.csv', index=False)
    print('spambase.csv', 'created')
# ---- Your program starts here ----

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv("spambase.csv")
print(df.head())
target_column = "class"
if target_column not in df.columns:
    raise ValueError("Replace class with the dataset's actual class/label column.")
df = df.dropna(subset=[target_column])
y = LabelEncoder().fit_transform(df[target_column].astype(str))
X = df.drop(columns=[target_column])
X = pd.get_dummies(X, drop_first=True)
X = X.replace([float("inf"), -float("inf")], pd.NA)
X = SimpleImputer(strategy="median").fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))

---
**Q3. Viva** (oral — no code)